# CoDiT quickstart: train + evaluate on the shipped `coastline` subset

This notebook trains a small CoDiT-GRU on a ~500-sample subset of the shipped
`coastline` merged training h5, then evaluates it on one held-out test patch.
Runtime target: less than 5 minutes on CPU.

Prerequisites: `CODIT_DATA_PATH` points at a directory containing the shipped
dataset (see `scripts/download_data.py`). See `docs/data_format.md` for the
processed-data layout.

In [ ]:
import os
from pathlib import Path

import jax
import numpy as np
from jax import numpy as jnp

from codit.data_module import config_funcs, config_paths
from codit.data_module.dataset import OceanDataset, OceanTrainingDataset
from codit.model_module.diffusion_model import CoDiTModel
from codit.eval_module.eval_runner import run_eval

np.random.seed(0)
key = jax.random.PRNGKey(0)
PREDICTION_LENGTH = 24

## 1. Load a small subset of the shipped training data

In [ ]:
processed = os.path.join(
    config_paths.DATA_PATH, 'datasets', 'coastline', 'datasets', 'processed'
)
assert Path(processed, f'training_data_{PREDICTION_LENGTH}h.h5').exists(), (
    'Shipped coastline training h5 not found — run scripts/download_data.py first.'
)

full = OceanTrainingDataset.from_disk(processed, PREDICTION_LENGTH)
full.remove_nans()
full.shuffle()

N_SUBSET = 500
full.trajectories = full.trajectories[:N_SUBSET]
full.context = full.context[:N_SUBSET]
full.indices = jnp.arange(N_SUBSET)

# val_dataloader in diffusion_model.py uses hardcoded batch_size=128, drop_last=True.
# Val set must have >=128 samples or training divides by zero.
train_ds, val_ds = full.split(0.7)
assert len(val_ds) >= 128, f'val set too small ({len(val_ds)}); increase N_SUBSET or lower split ratio'
train_ds.load_into_memory()
val_ds.load_into_memory()
len(train_ds), len(val_ds)

## 2. Instantiate CoDiT-GRU with quickstart-sized defaults

`K=20` (vs the shipped `K=100`) and 5 epochs keep the CPU wallclock inside
our ten-minute target.

In [ ]:
model = CoDiTModel(
    name='codit_GRU_quickstart',
    params={
        'prediction_length': PREDICTION_LENGTH,
        'dt': 1.0 / PREDICTION_LENGTH,
        'model_type': 'GRU',
        'context_encoder_type': 'conv',
        'K': 20,
        'vars': ['hycom_current', 'copernicus_current', 'era5_wind', 'bathymetry'],
    },
)
model.fit(
    train_ds,
    val_ds,
    train_params={
        'num_epochs': 5,
        'num_batches_per_epoch': 20,
        'batch_size': 32,
        'learning_rate': 1e-3,
        'optimizer': 'adam',
        'force_retrain': True,
        'early_stopping_patience': 100,
        'early_stopping_min_delta': 1e-6,
    },
)

## 3. Evaluate on one held-out patch

In [ ]:
config_path = os.path.join(config_paths.CONFIG_PATH, 'coastline', 'test')
patch_name = 'patch_1005_traj_300234066513050_1'
config = config_funcs.load_config(patch_name, config_path)
test_ds = OceanDataset(config, PREDICTION_LENGTH, 16, 10000, 'linear')
assert len(test_ds) > 0, f'Test patch {patch_name} produced 0 snippets'
distances, pred_coords, true_coords = run_eval(
    model,
    test_ds,
    indices=[0],  # single trajectory, keeps wallclock small
    length=PREDICTION_LENGTH,
    num_samples=10,
)
float(jnp.nanmean(distances)), pred_coords.shape

## 4. Plot the trajectory (true + one predicted sample)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(true_coords[0, :, 1], true_coords[0, :, 0], 'k-', label='true')
for s in range(pred_coords.shape[1]):
    ax.plot(pred_coords[0, s, :, 1], pred_coords[0, s, :, 0], 'C0-', alpha=0.3)
ax.set_xlabel('longitude')
ax.set_ylabel('latitude')
ax.set_title(f'{patch_name} — traj 0')
ax.legend()
plt.tight_layout()